In [1]:
# Setup: Add project root to Python path
import sys
from pathlib import Path

# Notebook is in examples/ subdirectory, go up one level to project root
project_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"[OK] Added to Python path: {project_root}")
print(f"[OK] Can now import 'materials' package")

[OK] Added to Python path: c:\Users\user\Repo\Scripts\section_design_checks
[OK] Can now import 'materials' package


In [2]:
# Materials library imports
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar
from materials.reinforced_concrete.constitutive import ConcreteModelType, SteelModelType
from materials.reinforced_concrete.analysis import create_interaction_diagram
from materials.reinforced_concrete.code_checks.ec2_2004 import BendingCheck
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_circular_section,
    create_linear_rebar_layer,
    create_circular_perimeter_rebars,
    RCSection,
)

In [3]:
# Create C30/37 concrete
concrete = ConcreteMaterial(grade="C30/37")

# Create B500B reinforcing steel
rebar_20 = Rebar(
    grade="B500B",
    diameter=20,      # mm
)

In [4]:
# Create rectangular section (centred at origin)
section_width = 300  # mm
section_height = 500  # mm

# Sections are created with their centre at (0,0)

rect_section = create_rectangular_section(
    width=section_width,
    height=section_height,
    section_name="Rectangular Beam",
)

# ϕ20 => radius = 10
cover = 50
r = rebar_20.diameter / 2

x_left  = cover + r   # 50 + 10 = 60
x_right =  section_width - cover - r   # 300 - 60 = 240

y_bottom = cover + r  # 50 + 10 = 60
y_top    =  section_height - cover - r  # 500 - 60 = 440

bottom_layer = create_linear_rebar_layer(
    rebar=rebar_20,
    n_bars=3,
    start_point=(x_left,  y_bottom),
    end_point=(x_right, y_bottom),
    layer_name="bottom",
)
rect_section.add_rebar_group(bottom_layer)

top_layer = create_linear_rebar_layer(
    rebar=rebar_20,
    n_bars=3,
    start_point=(x_left, y_top),
    end_point=(x_right, y_top),
    layer_name="top",
)
rect_section.add_rebar_group(top_layer)

In [5]:
rect_section.plot(
    concrete=concrete,
    show=False)


bending_check = BendingCheck(
    section=rect_section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.PARABOLA_RECTANGLE,
    steel_model_type=SteelModelType.HORIZONTAL
)

moment = 40
axial = 0

results = bending_check.perform_check(
    M_Ed=moment,
    N_Ed=axial
)

results.__str__()

#print(f"Utilisation: {results.utilization:.2f}")

'Bending check (EC2 §6.1): PASS (utilization: 24.1%) [N: 0.00/0.00 kN, M: 40.00/166.09 kN·m] - Section capacity adequate'

In [13]:
# Create M-N interaction diagram
rect_diagram = create_interaction_diagram(
    section=rect_section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.LINEAR_ELASTIC,  # EC2 Fig 3.3
    steel_model_type=SteelModelType.INCLINED,                  # With strain hardening
    n_fibres_width=20,
    n_fibres_height=30,
    include_tension=True,
    crack_to_neutral_axis_on_first_tension_failure=True
)

loadcases = []

moment = 40
axial = 0

forces_dict = {
    "M_Ed": moment,
    "N_Ed": axial
    }

loadcases.append(forces_dict)

rect_diagram.plot_mn(
    load_points=loadcases,
    show_vectors=True,
    show=True
    )

rect_diagram.plot_stress_strain(
    M_Ed=moment,
    N_Ed=axial,
    height=800,
    show=False,
    section_render="filled")

In [7]:
# Create circular section (centred at origin)
diameter = 500  # mm

# Sections are created with their centre at (0,0)
circular_section = create_circular_section(
    diameter=diameter,
    section_name="Circular Beam",
)

# ϕ20 => radius = 10
cover = 50

outer_layer = create_circular_perimeter_rebars(
    rebar=rebar_20,
    diameter=diameter,
    cover=cover,
    n_bars=12,
)
circular_section.add_rebar_group(outer_layer)

circular_section.plot(
    concrete=concrete,
    show=False
    )

In [8]:
# Create M-N interaction diagram
circular_diagram = create_interaction_diagram(
    section=circular_section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.PARABOLA_RECTANGLE,  # EC2 Fig 3.3
    steel_model_type=SteelModelType.INCLINED,                  # With strain hardening
    n_fibres_width=20,
    n_fibres_height=30,
)

loadcases = []

moment = 270
axial = 0

forces_dict = {
    "M_Ed": moment,
    "N_Ed": axial
    }

loadcases.append(forces_dict)

circular_diagram.plot_mn(
    load_points=loadcases,
    show_vectors=True,
    show=True
    )

circular_diagram.plot_stress_strain(
    M_Ed=moment,
    N_Ed=axial,
    height=800,
    show=False,
    section_render="filled")